# Análise Exploratória — Indicador Criança Alfabetizada

**Tech Challenge · Fase 2 — FIAP AI Scientist**

Este notebook percorre as fases do **CRISP-DM** aplicadas à camada Bronze do
projeto. O objetivo não é produzir a transformação — isso é papel do Glue Job
da camada Silver — mas **descobrir o que os dados realmente contêm** e
registrar as decisões que a Silver precisa tomar.

Cada achado aqui corresponde a uma decisão implementada em
`src/transformation/silver.py`.

| Fase CRISP-DM | Onde está |
|---|---|
| 1. Entendimento do negócio | Seção 1 |
| 2. Entendimento dos dados | Seções 2 a 4 — o corpo deste notebook |
| 3. Preparação dos dados | Seção 5 — o que a Silver faz com cada achado |
| 4. Modelagem | Fora do escopo desta fase |
| 5. Avaliação | Regras Q1–Q8, em `src/transformation/qualidade_silver.py` |
| 6. Implantação | Glue Workflow, declarado em `infra/terraform/` |

---

## 1. Entendimento do negócio

O Brasil assumiu o compromisso de alfabetizar todas as crianças até o final do
**2º ano do Ensino Fundamental até 2030**. O INEP aplica uma avaliação
padronizada e considera alfabetizado o estudante que atinge **743 pontos na
escala Saeb** de Língua Portuguesa.

**A pergunta que o dado precisa responder:** quais municípios estão abaixo da
meta, onde o avanço estagnou, e onde a intervenção pedagógica tem maior
retorno.

**A dificuldade não é obter o dado — é integrá-lo.** Resultados e metas são
tabelas distintas, com granularidades distintas, produzidas em momentos
distintos. Esta análise existe para descobrir *como* elas se conectam, antes
de escrever qualquer transformação.

**Critério de sucesso:** ao final, saber com precisão quais colunas ligam as
tabelas, quais valores precisam ser traduzidos, e quais ausências são
estruturais em vez de falhas.

### 2.1 Carga

Os dados vêm da camada Bronze, gravada pela ingestão a partir do dataset
público `basedosdados.br_inep_avaliacao_alfabetizacao` no BigQuery.

A leitura usa os arquivos locais em `data/bronze/` quando existem, e baixa do
S3 quando não — o que torna este notebook reproduzível por quem clonar o
repositório sem ter rodado a ingestão.

In [2]:
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

BUCKET = "fiap-ai-scientist-fase-02"

# Caminho relativo dentro da camada Bronze. Serve para o disco local e,
# prefixado por "bronze/", para a chave no S3.
TABELAS = {
    "uf": "alfabetizacao/uf.parquet",
    "municipio": "municipios/municipio.parquet",
    "alunos": "alunos/alunos.parquet",
    "meta_brasil": "metas_brasil/meta_alfabetizacao_brasil.parquet",
    "meta_uf": "metas_uf/meta_alfabetizacao_uf.parquet",
    "meta_municipio": "metas_municipios/meta_alfabetizacao_municipio.parquet",
    "dicionario": "dicionario/dicionario.parquet",
}

BRONZE_LOCAL = Path("../data/bronze")


def carregar(nome: str) -> pd.DataFrame:
    """
    Lê a tabela do disco; busca no S3 se não estiver local.

    A Bronze fica no S3 e não é versionada, então quem clonar o repositório
    não terá os arquivos. O download automático torna este notebook
    reproduzível sem exigir que a ingestão seja executada antes.
    """

    caminho = BRONZE_LOCAL / TABELAS[nome]

    if not caminho.exists():
        import boto3

        print(f"baixando {nome} do S3...")
        caminho.parent.mkdir(parents=True, exist_ok=True)
        boto3.client("s3").download_file(
            BUCKET, f"bronze/{TABELAS[nome]}", str(caminho)
        )

    return pd.read_parquet(caminho)


dados = {nome: carregar(nome) for nome in TABELAS}

print("Tabelas carregadas:\n")
for nome, df in dados.items():
    print(f"  {nome:16} {len(df):>10,} linhas x {len(df.columns):>2} colunas")

Tabelas carregadas:

  uf                      145 linhas x 15 colunas
  municipio            23,995 linhas x 15 colunas
  alunos            3,867,999 linhas x 12 colunas
  meta_brasil               3 linhas x 11 colunas
  meta_uf                  81 linhas x 12 colunas
  meta_municipio       10,704 linhas x 13 colunas
  dicionario               27 linhas x  5 colunas


### 2.2 Inventário

Primeira pergunta: o que cada tabela é, de fato. Os nomes sugerem uma coisa;
o conteúdo pode dizer outra.

In [3]:
inventario = pd.DataFrame([
    {
        "tabela": nome,
        "linhas": len(df),
        "colunas": len(df.columns),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 1),
        "duplicatas": int(df.duplicated().sum()),
    }
    for nome, df in dados.items()
]).sort_values("linhas", ascending=False)

inventario

,tabela,linhas,colunas,memoria_mb,duplicatas
2,alunos,3867999,12,466.8,0
1,municipio,23995,15,3.0,0
5,meta_municipio,10704,13,1.2,0
0,uf,145,15,0.0,0
4,meta_uf,81,12,0.0,0
6,dicionario,27,5,0.0,0
3,meta_brasil,3,11,0.0,0


**Achado 1 — `municipio` não é dimensão, é fato.**

O nome sugere um cadastro territorial. As colunas dizem outra coisa: são
`taxa_alfabetizacao`, `media_portugues` e as proporções por nível de
proficiência. É o **indicador** no grão município × ano × rede.

E há uma consequência: **este dataset não traz nome de município nem sigla de
UF**. A sigla é derivável dos dois primeiros dígitos do código IBGE; o nome
exigiria outra fonte.

In [4]:
dados["municipio"].head(3)

,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.3 O dicionário da própria fonte

A tabela `dicionario` é pequena e fácil de ignorar — e é a chave que decifra
os códigos das demais.

In [5]:
dic = dados["dicionario"]

for tabela in dic["id_tabela"].unique():
    for coluna in dic.loc[dic["id_tabela"] == tabela, "nome_coluna"].unique():
        recorte = dic[(dic["id_tabela"] == tabela) & (dic["nome_coluna"] == coluna)]
        print(f"\n[{tabela}].{coluna}")
        for _, linha in recorte.iterrows():
            print(f"    {linha['chave']:>3} = {linha['valor']}")


[alunos].alfabetizado
      0 = Não
      1 = Sim

[alunos].preenchimento_caderno
      0 = Prova não preenchida
      1 = Prova preenchida

[alunos].presenca
      0 = Ausente
      1 = Presente

[alunos].rede
      1 = Federal
      2 = Estadual
      3 = Municipal
      4 = Privada

[alunos].serie
      2 = 2° ano do Ensino Fundamental

[uf].rede
      0 = Total (Federal, Estadual, Municipal e Privada)
      1 = Federal
      2 = Estadual
      3 = Municipal
      4 = Privada
      5 = Pública (Estadual e Municipal)
      6 = Pública (Federal, Estadual e Municipal)

[uf].serie
      2 = 2° ano do Ensino Fundamental

[municipio].rede
      0 = Total (Federal, Estadual, Municipal e Privada)
      1 = Federal
      2 = Estadual
      3 = Municipal
      4 = Privada
      5 = Pública (Estadual e Municipal)
      6 = Pública (Federal, Estadual e Municipal)

[municipio].serie
      2 = 2° ano do Ensino Fundamental


### 2.4 A rede de ensino está codificada de duas formas

Este é o achado que decide se a integração é possível.

In [6]:
for nome in ["uf", "municipio", "alunos", "meta_brasil", "meta_uf", "meta_municipio"]:
    df = dados[nome]
    if "rede" in df.columns:
        valores = sorted(df["rede"].dropna().unique().tolist(), key=str)
        print(f"{nome:16} -> {valores}")

uf               -> ['0', '2', '3', '5']
municipio        -> ['0', '2', '3', '5']
alunos           -> ['2', '3', '4']
meta_brasil      -> ['Pública']
meta_uf          -> ['Pública']
meta_municipio   -> ['Municipal']


**Achado 2 — resultados usam código, metas usam texto.**

Os indicadores trazem `'0'`, `'2'`, `'3'`, `'5'`; as metas trazem `Municipal`
e `Pública`. Sem tradução, o join simplesmente não acontece.

O dicionário fornece a ponte:

| Código | Significado | Metas correspondentes |
|---|---|---|
| `3` | Municipal | `meta_alfabetizacao_municipio` |
| `5` | Pública (Estadual e Municipal) | `meta_alfabetizacao_uf`, `meta_alfabetizacao_brasil` |

Com isso, **uma linha de meta corresponde a exatamente uma de resultado** — o
join é 1:1, não 1:N.

### 2.5 As metas estão em formato largo

E o campo `ano` significa outra coisa do que parece.

In [7]:
brasil = dados["meta_brasil"]

colunas_meta = [c for c in brasil.columns if c.startswith("meta_alfabetizacao_")]

print("Colunas de meta:", colunas_meta)
print()
brasil[["ano", "taxa_alfabetizacao"] + colunas_meta]

Colunas de meta: ['meta_alfabetizacao_2024', 'meta_alfabetizacao_2025', 'meta_alfabetizacao_2026', 'meta_alfabetizacao_2027', 'meta_alfabetizacao_2028', 'meta_alfabetizacao_2029', 'meta_alfabetizacao_2030']



,ano,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030
0,2025,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0
1,2024,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0
2,2023,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0


**Achado 3 — `ano` é a safra de publicação, não o ano da meta.**

Cada linha traz sete colunas de meta, uma por ano-alvo. Comparar meta com
resultado exige transpor para formato longo.

**Achado 4 — as safras revisaram as metas.**

Repare que a safra de 2025 arredondou os alvos: a meta nacional de 2024 passou
de 59,9 para 60,0, e a de 2026 de 67,47 para 67,00.

In [8]:
divergentes = {
    coluna: sorted(brasil[coluna].dropna().unique().tolist())
    for coluna in colunas_meta
    if brasil[coluna].nunique() > 1
}

for coluna, valores in divergentes.items():
    print(f"{coluna}: {valores}")

print()
print("A serie de metas comeca em", min(int(c.split('_')[-1]) for c in colunas_meta))
print("2023 e o ano-base do Compromisso: nao existe meta para ele.")

meta_alfabetizacao_2024: [59.9, 60.0]
meta_alfabetizacao_2025: [63.77, 64.0]
meta_alfabetizacao_2026: [67.0, 67.47]
meta_alfabetizacao_2027: [70.97, 71.0]
meta_alfabetizacao_2028: [74.0, 74.23]
meta_alfabetizacao_2029: [77.0, 77.24]

A serie de metas comeca em 2024
2023 e o ano-base do Compromisso: nao existe meta para ele.


### 2.6 Os nulos são estruturais, não falhas

Quase metade das proporções por nível está vazia. A pergunta é se isso é
problema de dado ou característica da publicação.

In [9]:
mun = dados["municipio"]

niveis = [c for c in mun.columns if c.startswith("proporcao_aluno_nivel_")]

mun["tem_nivel"] = mun[niveis].notna().any(axis=1)

print("Presenca da distribuicao por nivel, por ano:\n")
print(pd.crosstab(mun["ano"], mun["tem_nivel"]))

print(f"\nNulos em {niveis[0]}: {mun[niveis[0]].isna().sum():,} "
      f"({mun[niveis[0]].isna().mean() * 100:.1f}%)")

Presenca da distribuicao por nivel, por ano:

tem_nivel  False  True 
ano                    
2023       11547      0
2024           0  12448

Nulos em proporcao_aluno_nivel_0: 11,547 (48.1%)


**Achado 5 — a distribuição por nível só existe em 2024.**

Os 48,1% de nulos correspondem integralmente ao ano de 2023. Não é falha: é
ausência estrutural de publicação.

Consequência para a análise: **a distribuição por nível de proficiência só
pode ser estudada para 2024.**

In [10]:
al = dados["alunos"]

resumo = al.groupby(["presenca", "preenchimento_caderno"], dropna=False)["proficiencia"].agg(
    total="size", nulos=lambda s: s.isna().sum()
)

print("Nulos de proficiencia por presenca e preenchimento:\n")
print(resumo)

print(f"\nTotal de nulos: {al['proficiencia'].isna().sum():,}")
print("Explicados sem sobra: ausentes + presentes com prova em branco.")

Nulos de proficiencia por presenca e preenchimento:

                                  total   nulos
presenca preenchimento_caderno                 
0        0                       512153  512153
1        0                         1185    1185
         1                      3354661       0

Total de nulos: 513,338
Explicados sem sobra: ausentes + presentes com prova em branco.


**Achado 6 — `proficiencia` é nula exatamente para quem não fez a prova.**

Não há nulo inexplicado. A ausência tem causa conhecida e registrada nas
próprias colunas de presença.

### 2.7 Verificação do ponto de corte

A documentação diz 743. Vale confirmar contra os dados, e não presumir.

In [11]:
PONTO_CORTE = 743

com_prof = al[al["proficiencia"].notna()].copy()

esperado = (com_prof["proficiencia"] >= PONTO_CORTE).astype(int)
declarado = com_prof["alfabetizado"].astype(int)

divergentes = int((esperado != declarado).sum())

print(f"Registros com proficiencia: {len(com_prof):,}")
print(f"Divergentes do corte {PONTO_CORTE}: {divergentes:,}")
print()
print("Regra confirmada empiricamente." if divergentes == 0
      else "ATENCAO: ha divergencia, investigar antes de assumir a regra.")

Registros com proficiencia: 3,354,661
Divergentes do corte 743: 0

Regra confirmada empiricamente.


**Achado 7 — a regra dos 743 é exata.**

Zero divergências em mais de 3,3 milhões de registros individuais. Não é
suposição da documentação: é fato medido.

### 2.8 A armadilha da coluna `alfabetizado`

Este é o achado que muda qualquer agregação feita a partir do nível estudante.

In [12]:
print("Valores distintos em alfabetizado:", sorted(al["alfabetizado"].unique()))
print("Nulos em alfabetizado:", int(al["alfabetizado"].isna().sum()))
print()

ausentes = al[al["presenca"] == "0"]
print(f"Alunos ausentes: {len(ausentes):,}")
print(f"  marcados como alfabetizado='0': {int((ausentes['alfabetizado'] == '0').sum()):,}")

# Comparacao das duas formas de calcular a taxa
ingenua = (al["alfabetizado"] == "1").mean() * 100

validos = al[(al["presenca"] == "1") & (al["preenchimento_caderno"] == "1")]
ponderada = (
    (validos["alfabetizado"] == "1") * validos["peso_aluno"]
).sum() / validos["peso_aluno"].sum() * 100

print()
print(f"Taxa calculada sem filtro:              {ingenua:.1f}%")
print(f"Taxa com alunos validos e ponderada:    {ponderada:.1f}%")
print(f"Diferenca:                              {ponderada - ingenua:.1f} pontos")

Valores distintos em alfabetizado: ['0', '1']
Nulos em alfabetizado: 0

Alunos ausentes: 512,153
  marcados como alfabetizado='0': 512,153

Taxa calculada sem filtro:              51.3%
Taxa com alunos validos e ponderada:    58.4%
Diferenca:                              7.1 pontos


**Achado 8 — ausentes constam como não alfabetizados.**

A coluna não tem nulos: quem não fez a prova aparece como `'0'`. Agregar
direto trata ausência como reprovação e produz número divergente do oficial.

**Regra obrigatória:** filtrar `presenca = 1` e `preenchimento_caderno = 1`, e
ponderar por `peso_aluno`.

---

## 3. Cobertura e lacunas

Antes de integrar, é preciso saber se as tabelas cobrem o mesmo universo.

In [13]:
MUNICIPIOS_BRASIL = 5570
UFS_BRASIL = 27

meta_mun = dados["meta_municipio"]

ids_resultado = set(mun["id_municipio"].unique())
ids_meta = set(meta_mun["id_municipio"].unique())

print(f"Municipios no Brasil (referencia):  {MUNICIPIOS_BRASIL:,}")
print(f"Com resultado:                      {len(ids_resultado):,}")
print(f"Com meta:                           {len(ids_meta):,}")
print(f"Com resultado e SEM meta:           {len(ids_resultado - ids_meta):,}")
print(f"Com meta e SEM resultado:           {len(ids_meta - ids_resultado):,}")
print(f"Ausentes das duas fontes:           {MUNICIPIOS_BRASIL - len(ids_resultado | ids_meta):,}")

ufs_resultado = set(dados["uf"]["sigla_uf"].unique())
ufs_meta = set(dados["meta_uf"]["sigla_uf"].unique())

print()
print(f"UFs com resultado: {len(ufs_resultado)} de {UFS_BRASIL}")
print(f"UFs ausentes:      {sorted(ufs_meta - ufs_resultado)}")

Municipios no Brasil (referencia):  5,570
Com resultado:                      5,550
Com meta:                           5,352
Com resultado e SEM meta:           198
Com meta e SEM resultado:           0
Ausentes das duas fontes:           20

UFs com resultado: 25 de 27
UFs ausentes:      ['DF', 'RR']


In [14]:
print("Cobertura temporal:\n")

for nome, df in dados.items():
    if "ano" in df.columns:
        print(f"  {nome:16} {sorted(df['ano'].dropna().unique().tolist())}")

Cobertura temporal:

  uf               [2023, 2024]
  municipio        [2023, 2024]
  alunos           [2023, 2024]
  meta_brasil      [2023, 2024, 2025]
  meta_uf          [2023, 2024, 2025]
  meta_municipio   [2023, 2024]


**Achado 9 — as coberturas não coincidem.**

- 198 municípios têm resultado e não têm meta publicada
- Nenhuma meta fica órfã: as metas são subconjunto dos resultados
- **DF e RR não aparecem na tabela por UF.** A ausência do DF é explicável —
  não possui rede municipal. A de RR fica registrada como lacuna conhecida
- Resultados cobrem 2023–2024; metas de UF e Brasil vão até 2025

Municípios sem meta **não devem ser descartados**: vão para quarentena com o
motivo registrado. Descarte silencioso faria um município desaparecer da
análise sem que ninguém percebesse.

---

## 4. Chaves e unicidade

Última verificação antes da preparação: as chaves candidatas são realmente
únicas?

In [15]:
candidatas = {
    "uf": ["ano", "sigla_uf", "rede"],
    "municipio": ["ano", "id_municipio", "rede"],
    "meta_uf": ["ano", "sigla_uf", "rede"],
    "meta_municipio": ["ano", "id_municipio", "rede"],
    "meta_brasil": ["ano", "rede"],
    "alunos": ["ano", "id_aluno"],
}

for nome, chave in candidatas.items():
    df = dados[nome]
    duplicadas = int(df.duplicated(subset=chave).sum())
    situacao = "UNICA" if duplicadas == 0 else f"{duplicadas:,} DUPLICADAS"
    print(f"{nome:16} {str(chave):42} -> {situacao}")

print()
print(f"alunos: {len(al):,} linhas para {al['id_aluno'].nunique():,} alunos distintos")
print("O mesmo estudante aparece em 2023 e 2024 — id_aluno sozinho nao e chave.")

uf               ['ano', 'sigla_uf', 'rede']                -> UNICA
municipio        ['ano', 'id_municipio', 'rede']            -> UNICA
meta_uf          ['ano', 'sigla_uf', 'rede']                -> UNICA
meta_municipio   ['ano', 'id_municipio', 'rede']            -> UNICA
meta_brasil      ['ano', 'rede']                            -> UNICA
alunos           ['ano', 'id_aluno']                        -> UNICA

alunos: 3,867,999 linhas para 2,352,328 alunos distintos
O mesmo estudante aparece em 2023 e 2024 — id_aluno sozinho nao e chave.


**Achado 10 — todas as chaves naturais são únicas.**

Nenhuma deduplicação é necessária. Isso é consequência direta da escolha da
fonte: a Base dos Dados já entrega o dado tratado. O esforço da Silver migra
de *consertar formato* para *resolver semântica*.

---

## 5. Preparação dos dados

Cada achado vira uma decisão implementada em
[`src/transformation/silver.py`](../src/transformation/silver.py).

| # | Achado | Decisão na Silver |
|---|---|---|
| 1 | `municipio` é fato, sem dimensão territorial | Derivar UF e região dos 2 primeiros dígitos do código IBGE |
| 2 | Rede em código e em texto | Traduzir texto → código antes do join (`Municipal`→`3`, `Pública`→`5`) |
| 3 | Metas em formato largo | Unpivot para grão ente × ano-alvo |
| 4 | Safras revisaram metas | Prevalece a safra mais recente |
| 5 | Distribuição por nível só em 2024 | Sinalizar com `tem_distribuicao_nivel`, nunca preencher com zero |
| 6 | Nulos de proficiência são estruturais | Marcar com `aluno_valido`, não filtrar linhas |
| 7 | Corte de 743 confirmado | Constante do contrato, verificada pela regra Q5 a cada execução |
| 8 | Ausentes contam como não alfabetizados | `fato_aluno` traz `aluno_valido` e `peso_aluno` |
| 9 | Coberturas divergentes | Classificar em `comparavel`, `ano_base`, `meta_nao_publicada`, `municipio_sem_meta` |
| 10 | Chaves únicas | Nenhuma deduplicação; unicidade virou a regra Q3 |

**Uma decisão que merece destaque:** `atingiu_meta` é booleano *nullable*. Onde
não há meta, o valor é nulo e não `False` — porque "não tem meta" e "não
atingiu a meta" são afirmações diferentes, e a segunda seria falsa sobre 216
municípios.

---

## 6. Próximas fases

**Modelagem.** A camada Gold recebe `meta_vs_resultado` no grão município ×
ano × rede, com 5.232 casos rotulados por `atingiu_meta` — base para
classificação de risco de não atingimento. E `fato_aluno` com
`faixa_proximidade` habilita a análise da massa imediatamente abaixo do corte,
o grupo com maior retorno marginal de intervenção.

**Avaliação.** As oito regras Q1–Q8 rodam como Glue Job a cada execução, com
relatório versionado. Regra bloqueante reprovada interrompe o Workflow e
impede a promoção para a Gold.

**Implantação.** Toda a infraestrutura é declarada em Terraform e orquestrada
por Glue Workflow: crawler → transformação → qualidade, encadeados dentro da
AWS.

---

**Ressalva de leitura.** Dois anos de cobertura sustentam comparação entre
anos, não afirmação de tendência. E o indicador mede um recorte específico da
alfabetização: usá-lo para ranquear e punir redes cria incentivo para otimizar
o número, não o aprendizado.